# Lesson 04 - टूल वापर डिझाइन पॅटर्न

या धड्यात तुम्ही Microsoft Agent Framework (Python) वापरून AI एजंटसाठी **टूल वापर** डिझाइन पॅटर्न शिकाल. आपण खालील गोष्टी समजावून घेऊ:

- `@tool` डेकॉरेटर आणि टायप केलेल्या पॅरामीटर्ससह फंक्शन टूल्स परिभाषित करणे
- टूल स्कीमा प्रदान करणे जेणेकरून मॉडेलला प्रत्येक टूल काय करते हे माहीत होईल
- `approval_mode` सह टूल कार्यान्वयन नियंत्रित करणे
- Pydantic मॉडेल्स आणि `response_format` द्वारे **रचनेतले आउटपुट** परत करणे

परिस्थिती ही एक **प्रवास बुकिंग एजंट** आहे जी स्थळे शोधू शकते, उपलब्धता तपासू शकते, आणि फ्लाइटची माहिती मिळवू शकते.


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटरसह साधने निश्चित करणे

`@tool` डेकोरेटर साध्या Python फंक्शनला एक साधन बनवते जे एजन्ट कॉल करू शकतो.
महत्वाचे मुद्दे:

- **डॉकस्टिंग** हा साधनाचा वर्णन बनतो जे मॉडेल पाहते.
- **टाइप अॅनोटेशन्स** (वर्णनांसह `Annotated` सहित) साधनाचे स्कीमा परिभाषित करतात.
- `approval_mode` कंट्रोल करते की वापरकर्त्याने प्रत्येक कॉलला त्याच्या अंमलबजावणीपूर्वी मंजुरी द्यावी की नाही.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## अनेक साधनांसह एजंट तयार करणे

मॉडेलला वापरकर्त्याच्या प्रश्नाचे उत्तर देण्यासाठी ज्या साधनांची गरज असेल, ते कोणतेही वापरू शकेल यासाठी सर्व तीन साधने क्लायंटला पाठवा.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## साधनांसह संरचित आउटपुट

`response_format` ला Pydantic मॉडेल म्हणून सेट केल्याने, एजंटला मुक्त-प्रारूप मजकूराऐवजी व्यवस्थित टाईप केलेले JSON ऑब्जेक्ट परत करणे आवश्यक आहे. जेव्हा डाउनस्ट्रीम कोडला परिणाम प्रोग्रामॅटिकली वापरणे आवश्यक असते तेव्हा हे उपयुक्त ठरते.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## टूल मंजुरी नमुने

`@tool` वरील `approval_mode` पॅरामीटर टूल कॉल्सना कार्यान्वित करण्यापूर्वी मानवी मंजुरी आवश्यक आहे की नाही हे नियंत्रित करते:

| मोड | वर्तन |
|---|---|
| `"never_require"` | टूल आपोआप चालते — वापरकर्त्याची पुष्टी आवश्यक नाही. |
| `"always_require"` | प्रत्येक कॉल कार्यान्वित होण्यापूर्वी वापरकर्त्याद्वारे मंजुरी दिली पाहिजे. |

ऐसे टूलसाठी `"always_require"` वापरा ज्यांचे साइड-इफेक्ट्स असतात (उदा. फ्लाइट बुक करणे, क्रेडिट कार्ड शुल्क लावणे) त्यामुळे मानवी हस्तक्षेप आवश्यक राहील.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## सारांश

या धड्यात तुम्ही खालील गोष्टी शिकल्या:

1. **@tool** डेकोरेटर वापरून टायप केलेल्या पॅरामीटर्स आणि टूल स्कीमा म्हणून कार्य करणाऱ्या डॉक्स्ट्रिंगसह टूल्स परिभाषित करणे.
2. **एकाधिक टूल्स संयोजित करणे** जेणेकरून एजंट त्यांना क्रमानुसार कॉल करून गुंतागुंतीच्या प्रश्नांना उत्तरे देऊ शकेल.
3. **रचनेत आउटपुट परत करणे** `response_format` म्हणून Pydantic मॉडेल पास करून.
4. **टूल मंजुरी नियंत्रण** `approval_mode` वापरून संवेदनशील क्रियांसाठी माणसाला ओळखात ठेवणे.

ही पद्धती विश्वसनीय, उत्पादनासाठी तयार एजंट तयार करण्यासाठी पाया घालतात जे सुरक्षितपणे बाह्य प्रणालींशी संवाद साधू शकतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
हा दस्तऐवज AI अनुवाद सेव्हिस [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केलेला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात ठेवा की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या स्थानिक भाषेतच अधिकारप्राप्त स्रोत मानला जावा. महत्त्वाच्या माहितींसाठी व्यावसायिक मानवी अनुवादाची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थ लावणाऱ्या परिस्थितींसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
